# Gaussian Mixture Models (GMM) Classification

I **Gaussian Mixture Models (GMM)** sono modelli generativi probabilistici in grado di approssimare distribuzioni di probabilità complesse e multimodali attraverso una combinazione convessa di $G$ componenti gaussiane multivariate:

$$f(x) = \sum_{g=1}^{G} w_g \, \mathcal{N}(x \mid \mu_g, \Sigma_g), \quad \text{con } \sum_{g=1}^{G} w_g = 1, \; w_g \geq 0$$

In questo notebook utilizziamo i GMM per modellare la densità di probabilità condizionata di ciascuna classe del dataset Iris ($P(x \mid C)$) ed effettuare la classificazione multiclasse mediante la regola di decisione a posteriori (Bayes Decision Rule).

In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy
import sklearn
from scripts.utils import *
from scripts.GMM import *

In [5]:
D, L, _, _ = init_dataset()
(DTR, LTR), (DVAL, LVAL) = split_db_2to1(D, L)

Le classi sono : ['setosa' 'versicolor' 'virginica'] e le features :['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


Per addestrare i GMM in modo stabile ed evitare problemi di singolarità nella covarianza (quando una gaussiana collassa su un singolo punto), abbiamo implementato una pipeline modulare in `scripts/GMM.py`:

1. **Valutazione congiunta (`logpdf_GMM`)**: Calcola le log-verosimiglianze congiunte $\log (w_g \mathcal{N}(x \mid \mu_g, \Sigma_g))$ e la densità marginale logaritmica sfruttando `scipy.special.logsumexp` per prevenire underflow numerico.
2. **Expectation-Maximization vincolato (`EM_constrained`)**:
   - **E-Step**: Calcolo delle *responsabilities* $\gamma_{ig} = P(g \mid x_i)$.
   - **M-Step**: Aggiornamento delle statistiche sufficienti ($Z_g, F_g, S_g$), pesi $w_g$, medie $\mu_g$ e matrici di covarianza $\Sigma_g$.
   - **Eigenvalue Flooring ($\psi$)**: Vincolo sugli autovalori di $\Sigma_g$ tramite SVD ($s_i = \max(s_i, \psi)$) per garantire che la matrice resti definita positiva e limitata inferiormente.
3. **Linde-Buzo-Gray Splitting (`LBG_constrained`)**: Inizializzazione gerarchica che parte da $G=1$ (gaussiana globale) e sdoppia le componenti lungo la direzione dell'autovettore principale ($d = \alpha \sqrt{\lambda_1} u_1$) fino a raggiungere il numero target di componenti $G \in \{1, 2, 4, 8, 16\}$.

Di seguito addestriamo un modello GMM indipendente per ciascuna classe sui dati di training e valutiamo l'**Error Rate** sul validation set al variare del numero di componenti.

In [6]:
n_components = [1, 2, 4, 8, 16]

# 1. Separazione del dataset di training per classe
DTR0 = DTR[:, LTR == 0]
DTR1 = DTR[:, LTR == 1]
DTR2 = DTR[:, LTR == 2]

for i in n_components:
    print(f"GMM Components: {i}")
    
    # 2. Addestramento di un GMM SEPARATO per ogni singola classe
    gmm0 = LBG_constrained(DTR0, threshold=1e-6, target_components=i, alpha=0.1, psi=0.01)
    gmm1 = LBG_constrained(DTR1, threshold=1e-6, target_components=i, alpha=0.1, psi=0.01)
    gmm2 = LBG_constrained(DTR2, threshold=1e-6, target_components=i, alpha=0.1, psi=0.01)
    # 3. Calcolo delle log-densità marginali sul validation set (ignoriamo la matrice S)
    _, logdens0 = logpdf_GMM(DVAL, gmm0)
    _, logdens1 = logpdf_GMM(DVAL, gmm1)
    _, logdens2 = logpdf_GMM(DVAL, gmm2)

    # 4. Calcolo delle Log-Likelihood Ratios (LLR per classificazione binaria)
    
    SPost=np.vstack([logdens0,logdens1,logdens2])
    
    # 5. Predizione: se LLR > 0 vince la classe 1, altrimenti classe 0
    predictions = np.argmax(SPost,axis=0)
    
    # Calcolo dell'accuratezza e dell'Error Rate
    acc = np.where(predictions == LVAL, 1, 0).mean()
    err = 1 - acc
    
    # Stampa formattata per maggiore chiarezza
    print(f"Error rate: {err * 100:.2f}%\n")

GMM Components: 1
Error rate: 4.00%

GMM Components: 2
Error rate: 4.00%

GMM Components: 4
Error rate: 4.00%

GMM Components: 8
Error rate: 4.00%

GMM Components: 16
Error rate: 4.00%



- **Performance con $G=1$**: Il modello a 1 componente corrisponde a un classico classificatore Gaussiano Tied/Untied; sul validation set raggiunge già un error rate competitivo pari al **4.00%** (corrispondente a 2 soli campioni errati su 50).
- **Stabilità al crescere delle componenti**: Grazie all'introduzione dell'eigenvalue flooring ($\psi = 0.01$) e alla regolarizzazione LBG, l'algoritmo EM non soffre di degenerazione della matrice di covarianza anche aumentando le componenti fino a $G = 16$.